# GNN Model Implementation
> **Data Requirement:** This model requires `Data Files/negative_ProjectFinal_filtered_score0.1.csv`. 
> If you haven't generated it using the `SequenceIntegration` notebook, please download it [HERE](https://drive.google.com/file/d/1Aaw5wcOaLjBBvab3sXhod25xtk9jmJUp/view?usp=drive_link).

The following cell creates the **custom negative set of pure negatives**, the one with 60/40 split logic.

In [ ]:
import pandas as pd
import numpy as np

# ===============================
# Configuration & File paths.
# ===============================

# The main positive and negative datasets
POS_FILE = r"Data Files\positive_ProjectFinal.csv"
FULL_NEG_FILE = r"Data Files\negative_ProjectFinal_filtered_score0.1.csv"

# Output file name
OUTPUT_NEG_FILE = "Total_Sampled_CustomNegative.csv"

# Sampling Parameters
NEG_SAMPLE_SEED = 42 # Seed used for reproducibility
NODE_SAMPLE_SEED = 13 # Seed which will beused for the 60/40 node split
np.random.seed(NODE_SAMPLE_SEED)

# List of all required columns from the datasets.
ALL_COLUMNS = ['Human', 'Human_Sequence', 'SARS-CoV2', 'SARS-CoV2_Sequence', 'Score']

# ======================
# 1. Loading data files
# ======================

print("Loading data files...")
try:
    # Load only the specified columns to ensure consistency
    pos_df = pd.read_csv(POS_FILE, usecols=ALL_COLUMNS)
    full_neg_df = pd.read_csv(FULL_NEG_FILE, usecols=ALL_COLUMNS)
except FileNotFoundError as e:
    print(f"\n--- ERROR ---: File not found: {e.filename}")
    print("Please ensure your data files are named correctly and are in the same directory.")
    exit()

# =====================================================
# 2. Creating the custom negative set (Pure Negatives)
# =====================================================

print("\nCreating the Custom Negative Set (Pure Negatives)...")

# -- Step 1: At first we clean the negative datset by removing overlaps with the positive dataset --
# Then creating a set of all positive pairs for fast lookup.
# Here the sorting ensures that(A, B) and (B, A) are treated as the same pair
pos_pairs = set(pos_df.apply(lambda x: tuple(sorted((x['Human'], x['SARS-CoV2']))), axis=1))

def is_positive_overlap(row):
    """Checks if a negative pair exists in the positive set (should be rare)."""
    pair = tuple(sorted((row['Human'], row['SARS-CoV2'])))
    return pair in pos_pairs

# Now removing overlaps
initial_neg_size = len(full_neg_df)
# The full set of columns is carried through this filtering step
neg_df_cleaned = full_neg_df[~full_neg_df.apply(is_positive_overlap, axis=1)].reset_index(drop=True)
print(f"Removed {initial_neg_size - len(neg_df_cleaned)} overlaps. Clean negative pool size: {len(neg_df_cleaned)}")
N_TEST_EDGES = len(neg_df_cleaned)

# -- Step 2: Node Sampling  --
# Nodes found in the positive training data
pos_nodes_human = set(pos_df['Human'].unique())
pos_nodes_viral = set(pos_df['SARS-CoV2'].unique())

# Nodes only found in the cleaned negative data (i.e., not in the positive data)
all_nodes_human = set(neg_df_cleaned['Human'].unique())
all_nodes_viral = set(neg_df_cleaned['SARS-CoV2'].unique())

neg_only_nodes_human = list(all_nodes_human - pos_nodes_human)
neg_only_nodes_viral = list(all_nodes_viral - pos_nodes_viral)

# Defining sample size ratios which is the 60/40 split that we intended to do.


# Sample Human Nodes: 60% from positive context, 40% from negative-only context
sampled_human_nodes = (
    np.random.choice(list(pos_nodes_human), size=int(len(pos_nodes_human) * 0.6), replace=False).tolist() +
    np.random.choice(neg_only_nodes_human, size=int(len(neg_only_nodes_human) * 0.4), replace=False).tolist()
)

# Sample Viral Nodes: 60% from positive context, 40% from negative-only context
sampled_viral_nodes = (
    np.random.choice(list(pos_nodes_viral), size=int(len(pos_nodes_viral) * 0.6), replace=False).tolist() +
    np.random.choice(neg_only_nodes_viral, size=int(len(neg_only_nodes_viral) * 0.4), replace=False).tolist()
)

sampled_human_nodes_set = set(sampled_human_nodes)
sampled_viral_nodes_set = set(sampled_viral_nodes)
print(f"Subgraph boundary defined with {len(sampled_human_nodes)} Human and {len(sampled_viral_nodes)} Viral nodes.")


# -- Step 3: Edge Filtering i.e. finding edges within the subgraph --
# 3.1. Sampling a large pool of negative candidates from the cleaned negative set
test_neg_edges_df_candidates = neg_df_cleaned.sample(n=N_TEST_EDGES, random_state=NEG_SAMPLE_SEED, replace=False).reset_index(drop=True)

# 3.2. Define the filtering function
def is_in_sampled_subgraph(row):
    """Checks if both nodes in the pair are within the sampled node boundary."""
    human_node = row['Human']
    viral_node = row['SARS-CoV2']
    return (human_node in sampled_human_nodes_set) and (viral_node in sampled_viral_nodes_set)

# 3.3. Now we apply the filter to generate the final custom set
# The full set of columns is carried through this final filtering step
custom_neg_set_836 = test_neg_edges_df_candidates[test_neg_edges_df_candidates.apply(is_in_sampled_subgraph, axis=1)].reset_index(drop=True)

# Verifying the size
print(f"Final Custom Negative Set generated successfully. Total edges: {len(custom_neg_set_836)}")
custom_neg_set_836.to_csv(OUTPUT_NEG_FILE, index=False)
print(f"Saved Custom Negative Sample ({len(custom_neg_set_836)} rows) to: {OUTPUT_NEG_FILE}")

Loading data files...

Replicating the Custom Negative Set (Pure Negatives)...
Removed 3124 overlaps. Clean negative pool size: 172966
Subgraph boundary defined with 7711 Human and 7 Viral nodes.
Final Custom Negative Set generated successfully. Total edges: 45034
Saved Custom Negative Sample (45034 rows) to: Total_Sampled_CustomNegative.csv


The **GNN** model

In [ ]:
# =======================
# 1. Necessary imports.
# =======================
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc
)
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================
# 2. Loading ProtBERT model
# ==========================
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
protbert = BertModel.from_pretrained("Rostlab/prot_bert").to(device)
protbert.eval()

MAX_LEN = 512

def embed_protein(seq):
    """Chunk-safe ProtBERT embedding"""
    seq = seq.replace(" ", "").upper()
    chunks = [seq[i:i+MAX_LEN] for i in range(0, len(seq), MAX_LEN)]

    reps = []
    for ch in chunks:
        spaced = " ".join(list(ch))
        inputs = tokenizer(spaced, return_tensors="pt").to(device)
        with torch.no_grad():
            out = protbert(**inputs)
        reps.append(out.last_hidden_state[:,0,:])   # CLS token

    rep = torch.mean(torch.stack(reps), dim=0)      # Mean pool chunks
    return rep.squeeze(0)

# ======================
# 3. Loading data files
# ======================
pos_df = pd.read_csv(r"Data Files/positive_ProjectFinal.csv")
neg_df = pd.read_csv("Total_Sampled_CustomNegative.csv")

# One negative set
neg_df = neg_df.sample(n=len(pos_df), random_state=42).reset_index(drop=True)

df = pd.concat([pos_df, neg_df], ignore_index=True)
df["Label"] = [1]*len(pos_df) + [0]*len(neg_df)

# ===============================
# 4. Building node dictionary
# ===============================
all_humans = df["Human"].unique().tolist()
all_viruses = df["SARS-CoV2"].unique().tolist()

human_to_id = {h: i for i, h in enumerate(all_humans)}
virus_to_id = {v: i+len(all_humans) for i, v in enumerate(all_viruses)}

num_nodes = len(all_humans) + len(all_viruses)

# ===========================
# 5. Embedding all proteins
# ===========================
print("\nEmbedding all proteins...")

node_features = torch.zeros((num_nodes, 1024), device=device)

for h in all_humans:
    node_features[human_to_id[h]] = embed_protein(
        df[df["Human"] == h]["Human_Sequence"].iloc[0]
    )

for v in all_viruses:
    node_features[virus_to_id[v]] = embed_protein(
        df[df["SARS-CoV2"] == v]["SARS-CoV2_Sequence"].iloc[0]
    )

# ========================
# 6. Building graph edges
# ========================
edge_list = []

for i, row in df.iterrows():
    u = human_to_id[row["Human"]]
    v = virus_to_id[row["SARS-CoV2"]]
    edge_list.append([u, v])
    edge_list.append([v, u])   # This is an undirected graph

edge_index = torch.tensor(edge_list, dtype=torch.long).t().to(device)

# Edge pairs which will be used for GNN classification
edge_pairs = torch.tensor([
    [human_to_id[row["Human"]], virus_to_id[row["SARS-CoV2"]]]
    for _, row in df.iterrows()
], dtype=torch.long).to(device)

labels = torch.tensor(df["Label"].values).long().to(device)

# ==================
# 7. The GCN model
# ==================
class GCN(nn.Module):
    def __init__(self, in_dim=1024, hid=512, out_dim=256):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hid)
        self.conv2 = GCNConv(hid, out_dim)
        self.relu = nn.ReLU()

        self.edge_mlp = nn.Sequential(
            nn.Linear(out_dim*2, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x, edge_index, edge_pairs):
        x = self.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        u = x[edge_pairs[:,0]]
        v = x[edge_pairs[:,1]]
        uv = torch.cat([u, v], dim=1)

        return self.edge_mlp(uv).squeeze()

# ===============================
# 8. The K-fold training with k=5
# ===============================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

final_metrics = []
all_probs = []
all_true = []

for fold, (train_i, test_i) in enumerate(kf.split(edge_pairs)):
    print(f"\n===== FOLD {fold+1} =====")

    model = GCN().to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCELoss()

    train_idx = torch.tensor(train_i).to(device)
    test_idx = torch.tensor(test_i).to(device)

    # - Train -
    for epoch in range(8):
        model.train()
        opt.zero_grad()

        out = model(node_features, edge_index, edge_pairs[train_idx])
        loss = loss_fn(out, labels[train_idx].float())
        loss.backward()
        opt.step()

        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    # - Test -
    model.eval()
    with torch.no_grad():
        probs = model(node_features, edge_index, edge_pairs[test_idx]).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        true = labels[test_idx].cpu().numpy()

        # Saving the combined curves
        all_probs.extend(probs)
        all_true.extend(true)

        # Folding metrics
        acc = accuracy_score(true, preds)
        prec = precision_score(true, preds)
        rec = recall_score(true, preds)
        f1 = f1_score(true, preds)
        auc_val = roc_auc_score(true, probs)

        p_vals, r_vals, _ = precision_recall_curve(true, probs)
        pr_auc = auc(r_vals, p_vals)

        final_metrics.append([acc, prec, rec, f1, auc_val, pr_auc])

# ===================================================
# 9. The combined ROC and PR curves across all folds
# ===================================================
all_probs = np.array(all_probs)
all_true = np.array(all_true)

# ---- Final combined ROC ----
fpr, tpr, _ = roc_curve(all_true, all_probs)
roc_auc_val = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_val:.4f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("FINAL COMBINED ROC CURVE (Across 5 Folds)")
plt.legend()
plt.show()

# ---- Final combined PR ----
precision_vals, recall_vals, _ = precision_recall_curve(all_true, all_probs)
pr_auc_val = auc(recall_vals, precision_vals)

plt.plot(recall_vals, precision_vals, label=f"AUPRC = {pr_auc_val:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("FINAL COMBINED PR CURVE (Across 5 Folds)")
plt.legend()
plt.show()

# ===============================================
# 10. Final mmetrices averaged across all 5 folds
# ===============================================
m = np.array(final_metrics)

print("\n===== FINAL RESULTS (AVERAGED ACROSS 5 FOLDS) =====")
print(f"Accuracy:   {m[:,0].mean():.4f}")
print(f"Precision:  {m[:,1].mean():.4f}")
print(f"Recall:     {m[:,2].mean():.4f}")
print(f"F1 Score:   {m[:,3].mean():.4f}")
print(f"AUC-ROC:    {m[:,4].mean():.4f}")
print(f"AUPRC:      {m[:,5].mean():.4f}")

